# A gala concierge grounded in the course guest list

The concierge has two decisions to make. A general question about the gala goes to the guest list. If the first search finds no match, the concierge tries a small vocabulary of related terms. It returns a passage and a guest identifier so the host can check the suggestion. The guest descriptions are fictional course records.

[Unit 3 assignment](https://huggingface.co/learn/agents-course/en/unit3/agentic-rag/introduction) · [Guest dataset](https://huggingface.co/datasets/agents-course/unit3-invitees)

In [1]:
import re
import requests

dataset = 'agents-course/unit3-invitees'
params = dict(dataset=dataset, config='default', split='train', offset=0, length=100)
response = requests.get('https://datasets-server.huggingface.co/rows', params=params, timeout=20)
response.raise_for_status()
guests = [{'id': r['row_idx'], 'name': r['row']['name'], 'text': r['row']['description']}
          for r in response.json()['rows']]
assert guests and len({g['id'] for g in guests}) == len(guests)
print('Public guest records:', len(guests))
print(', '.join(g['name'] for g in guests))

Public guest records: 3
Ada Lovelace, Dr. Nikola Tesla, Marie Curie


## Retrieve, inspect and decide

Rank each record by the number of distinct search terms that occur in its name or description. Return all equally strong matches so an arbitrary tie cannot turn into a recommendation. The second search uses a fixed set of synonyms only when the first search finds no evidence.

In [2]:
stopwords = set('a an the is was do does did who which whom with likes would should i my to of in on me discuss ask talk about used for and at'.split())
related = {'coding': {'programmer', 'computing'}, 'birds': {'pigeons'},
           'radiation': {'radioactivity'}, 'power': {'energy'}}

def terms(text):
    return set(re.findall(r'[a-z]+', text.lower())) - stopwords

def retrieve(query_terms):
    scored = [(len(query_terms & terms(g['name'] + ' ' + g['text'])), g) for g in guests]
    best = max(score for score, _ in scored)
    return [g for score, g in scored if best and score == best]

def concierge(question):
    query = terms(question)
    matches = retrieve(query)
    trace = ['guest search']
    if not matches:
        extra = set().union(*(related.get(term, set()) for term in query))
        if extra:
            matches = retrieve(extra)
            trace.append('expanded search: ' + ', '.join(sorted(extra)))
    if len(matches) != 1:
        return {'status': 'ambiguous' if matches else 'no evidence',
                'candidates': [g['name'] for g in matches], 'steps': trace}
    g = matches[0]
    return {'status': 'matched', 'guest': g['name'], 'evidence': g['text'],
            'source': f'https://huggingface.co/datasets/{dataset}/viewer/default/train?row={g["id"]}',
            'steps': trace}

result = concierge('Who would discuss birds with me?')
print(result['steps'])
print(result['guest'], result['evidence'], result['source'], sep='\n')

['guest search', 'expanded search: pigeons']
Dr. Nikola Tesla
Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
https://huggingface.co/datasets/agents-course/unit3-invitees/viewer/default/train?row=1


In [3]:
checks = [('Who worked on the Analytical Engine?', 'Ada Lovelace'),
          ('Whom should I ask about radiation?', 'Marie Curie'),
          ('Who would discuss birds with me?', 'Dr. Nikola Tesla')]
for question, expected in checks:
    result = concierge(question)
    source_record = next(g for g in guests if g['name'] == expected)
    assert result['guest'] == expected and result['evidence'] == source_record['text']
    assert result['source'].endswith(f'row={source_record["id"]}')
    print(f'{question} -> {expected} ({len(result["steps"])} search step(s))')
missing = concierge('Whom should I ask about glaciology?')
assert missing['status'] == 'no evidence'
print('Glaciology:', missing['status'])

Who worked on the Analytical Engine? -> Ada Lovelace (1 search step(s))
Whom should I ask about radiation? -> Marie Curie (2 search step(s))
Who would discuss birds with me? -> Dr. Nikola Tesla (2 search step(s))
Glaciology: no evidence


The bird question needed a second search for *pigeons*. The glaciology question exhausted the available evidence. This small check demonstrates routing and attribution on three public records. A broader guest list would need more paraphrase and ambiguity tests. The notebook uses public read access and requires internet access to refresh the guest list.